# CineData Analytics | Silver → Gold

**Entregas desta camada:**
1. **Star Schema** para o time de BI: `fact_movies_performance`, dimensões (`dim_movies`, `dim_genres`,
   `dim_people`, `dim_companies`, `dim_reviews`) e tabelas-ponte (`bridge_movie_genre`, `bridge_movie_person`,
   `bridge_movie_company`).
2. **`gold_genai_movies_context`** para o Vector Search do assistente de IA (RAG).
3. **Desafio de Analytics**: 6 perguntas de negócio respondidas com `display()`.

**Surrogate Keys:** geradas por **hash SHA-256** da chave natural (primeiros 15 dígitos hexadecimais → BIGINT).
Diferente de `monotonically_increasing_id()`, o hash é **determinístico**: o mesmo filme recebe a mesma SK em
toda execução, então fato, dimensões e pontes continuam consistentes mesmo com reprocessamentos.

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [ ]:
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo").strip()

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Camada Gold - modelo dimensional e produtos de dados'")

# Decisão de escopo da fato: o enunciado pede métricas de "filmes lançados". Filmes planejados/em produção
# ficam na dim_movies (e no contexto da IA), mas não entram na fato de performance.
FATO_APENAS_LANCADOS = True
TIPOS_PESSOA = ["Ator", "Diretor", "Roteirista"]

In [ ]:
def gerar_sk(*colunas: str):
    """Surrogate key determinística: sha2(chave natural) -> 15 hex (60 bits) -> BIGINT positivo."""
    chave = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in colunas])
    return F.conv(F.substring(F.sha2(chave, 256), 1, 15), 16, 10).cast("bigint")


def validar_unicidade(df: DataFrame, colunas: list, nome: str) -> None:
    """Garante o grão declarado: falha o job se a chave se repetir."""
    total, distintos = df.count(), df.select(*colunas).distinct().count()
    if total != distintos:
        raise AssertionError(f"{nome}: grão violado ({total} linhas x {distintos} chaves distintas)")


def salvar(df: DataFrame, tabela: str, chave: list = None) -> None:
    if chave:
        validar_unicidade(df, chave, tabela)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{tabela:<35} | {spark.table(tabela).count():>8} linhas")

## Entrega 1 — Star Schema
### Dimensões

In [ ]:
# gold.dim_movies — metadados descritivos (1 linha por filme; a Silver já garante unicidade)
dim_movies = (
    spark.table("silver.tb_info_filmes")
    .select(
        gerar_sk("id_filme").alias("sk_movie_id"),
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)
salvar(dim_movies, "gold.dim_movies", ["sk_movie_id"])

# gold.dim_genres — catálogo único de gêneros
dim_genres = (
    spark.table("silver.tb_generos").select("nome_genero").distinct()
    .select(gerar_sk("nome_genero").alias("sk_genre_id"), F.col("nome_genero").cast("string"))
)
salvar(dim_genres, "gold.dim_genres", ["sk_genre_id"])

# gold.dim_people — somente pessoas físicas. A mesma pessoa pode ter 2 papéis (ex.: atua e dirige):
# nesse caso são 2 linhas, pois o papel (tipo_pessoa) faz parte do registro da dimensão.
pessoas_empresas = spark.table("silver.tb_pessoas_empresas")
dim_people = (
    pessoas_empresas.filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct()
    .select(gerar_sk("nome_pessoa", "tipo_pessoa").alias("sk_person_id"), "nome_pessoa", "tipo_pessoa")
)
salvar(dim_people, "gold.dim_people", ["sk_person_id"])

# gold.dim_companies — catálogo único de produtoras/estúdios
dim_companies = (
    pessoas_empresas.filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora")).distinct()
    .select(gerar_sk("nome_produtora").alias("sk_company_id"), "nome_produtora")
)
salvar(dim_companies, "gold.dim_companies", ["sk_company_id"])

In [ ]:
# gold.dim_reviews — avaliações individuais resumidas por filme (1 linha por filme avaliado).
# qtd_avaliacoes_usuarios conta todas as avaliações; a média ignora notas NULL (fora da escala na origem).
dim_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")   # FK válida: só filmes da dimensão
    .withColumn("_prefixo", F.lit("review"))   # prefixo evita que sk_review_id seja igual ao sk_movie_id
    .select(gerar_sk("_prefixo", "id_filme").alias("sk_review_id"),
            "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
salvar(dim_reviews, "gold.dim_reviews", ["sk_review_id"])

### Tabela Fato — `gold.fact_movies_performance`
**Grão:** 1 linha por filme. As tabelas Silver financeira e de métricas já são únicas por `id_filme`, por isso os
`LEFT JOIN` a partir da dimensão **não multiplicam** linhas (validado abaixo). Gêneros, pessoas e produtoras
**não** entram na fato: ficam nas pontes, justamente para não duplicar o grão.

In [ ]:
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

base_fato = dim_movies.filter(F.col("status_filme") == "Lançado") if FATO_APENAS_LANCADOS else dim_movies

fact = (
    base_fato.select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)
salvar(fact, "gold.fact_movies_performance", ["sk_movie_id"])

### Tabelas-ponte (N:N)
Um filme tem vários gêneros/pessoas/produtoras e vice-versa. As pontes ligam `dim_movies` às dimensões periféricas
sem tocar no grão da fato. `bridge_movie_person` carrega também `ordem_credito` (posição no elenco da origem),
usada para identificar os **atores principais** no documento da IA.

In [ ]:
sk_filmes = dim_movies.select("id_filme", "sk_movie_id")

bridge_genre = (
    spark.table("silver.tb_generos")
    .join(sk_filmes, "id_filme")
    .join(dim_genres, "nome_genero")
    .select("sk_movie_id", "sk_genre_id").distinct()
)
salvar(bridge_genre, "gold.bridge_movie_genre", ["sk_movie_id", "sk_genre_id"])

bridge_person = (
    pessoas_empresas.filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .withColumnRenamed("nome_entidade", "nome_pessoa").withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .join(sk_filmes, "id_filme")
    .join(dim_people, ["nome_pessoa", "tipo_pessoa"])
    .groupBy("sk_movie_id", "sk_person_id")
    .agg(F.min("ordem_credito").cast("int").alias("ordem_credito"))
)
salvar(bridge_person, "gold.bridge_movie_person", ["sk_movie_id", "sk_person_id"])

bridge_company = (
    pessoas_empresas.filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .join(sk_filmes, "id_filme")
    .join(dim_companies, "nome_produtora")
    .select("sk_movie_id", "sk_company_id").distinct()
)
salvar(bridge_company, "gold.bridge_movie_company", ["sk_movie_id", "sk_company_id"])

## Entrega 2 — `gold.gold_genai_movies_context`

**A "casca de banana" dos nulos.** `concat()` devolve NULL se **qualquer** parte for NULL, e o filme sumiria do
contexto da IA sem erro nenhum. Avaliação feita campo a campo (célula abaixo mostra a contagem de nulos):

| Campo | Pode vir nulo? | Fallback (via `coalesce`) |
|---|---|---|
| Título | sim (Column Shift) | `"sem título informado"` |
| Ano | sim (data impossível de converter) | a frase vira `"com ano de lançamento não informado"` |
| Receita / Orçamento | muito comum (valores 0/Unknown viraram NULL; filmes não lançados não estão na fato) | `"um valor não informado"` |
| Atores | sim (elenco vazio na origem) | `"elenco não informado"` |
| Diretor | sim | `"direção não informada"` |
| Sinopse | sim | `"sinopse não disponível"` |

Usamos `coalesce(campo, fallback)` em **cada** peça antes do `concat`, e validamos no final que nenhum documento
ficou nulo e que todos os filmes da `dim_movies` estão presentes.

In [ ]:
display(dim_movies.select(*[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}")
                            for c in ["titulo", "ano_lancamento", "sinopse"]]))
display(fact.select(*[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}")
                      for c in ["receita_usd", "orcamento_usd"]]))

In [ ]:
def lista_nomes_por_filme(tipo: str, limite: int, alias: str) -> DataFrame:
    """Agrega os nomes de um tipo de pessoa por filme em uma frase: 'A, B e C' (ordem de crédito da origem)."""
    return (
        bridge_person.join(dim_people.filter(F.col("tipo_pessoa") == tipo), "sk_person_id")
        .groupBy("sk_movie_id")
        .agg(F.expr(f"slice(transform(array_sort(collect_list(struct(ordem_credito, nome_pessoa))), "
                    f"x -> x.nome_pessoa), 1, {limite})").alias("_lista"))
        .select("sk_movie_id", F.expr(
            "CASE WHEN size(_lista) <= 1 THEN _lista[0] "
            "ELSE concat(array_join(slice(_lista, 1, size(_lista) - 1), ', '), ' e ', element_at(_lista, -1)) END"
        ).alias(alias))
    )


def valor_monetario(usd: str, brl: str):
    """'US$ 1,234,567.00 (R$ 6,543,210.00)' — NULL se o valor não existir (tratado pelo coalesce depois)."""
    return F.concat(F.lit("US$ "), F.format_number(F.col(usd), 2), F.lit(" (R$ "), F.format_number(F.col(brl), 2), F.lit(")"))


atores = lista_nomes_por_filme("Ator", 5, "atores")          # 5 primeiros créditos = atores principais
diretores = lista_nomes_por_filme("Diretor", 3, "diretores")

sinopse_limpa = F.regexp_replace(F.col("sinopse"), r"[\s.]+$", "")   # evita '..' no final do documento
trecho_ano = F.when(F.col("ano_lancamento").isNotNull(),
                    F.concat(F.lit("lançado no ano de "), F.col("ano_lancamento").cast("string"))
                    ).otherwise(F.lit("com ano de lançamento não informado"))

genai = (
    dim_movies
    .join(fact.select("sk_movie_id", "receita_usd", "receita_brl", "orcamento_usd", "orcamento_brl"), "sk_movie_id", "left")
    .join(atores, "sk_movie_id", "left")
    .join(diretores, "sk_movie_id", "left")
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        F.concat(
            F.lit("O filme "), F.coalesce(F.col("titulo"), F.lit("sem título informado")),
            F.lit(", "), trecho_ano,
            F.lit(", faturou "), F.coalesce(valor_monetario("receita_usd", "receita_brl"), F.lit("um valor não informado")),
            F.lit(" e teve um custo de "), F.coalesce(valor_monetario("orcamento_usd", "orcamento_brl"), F.lit("um valor não informado")),
            F.lit(". Estrelado por "), F.coalesce(F.col("atores"), F.lit("elenco não informado")),
            F.lit(" e dirigido por "), F.coalesce(F.col("diretores"), F.lit("direção não informada")),
            F.lit(", o filme possui a seguinte sinopse: "), F.coalesce(sinopse_limpa, F.lit("sinopse não disponível")),
            F.lit("."),
        ).alias("llm_context_document"),
    )
)
salvar(genai, "gold.gold_genai_movies_context", ["movie_id"])

# Change Data Feed habilitado: requisito do Databricks Vector Search (Delta Sync Index) para sincronização incremental
spark.sql("ALTER TABLE gold.gold_genai_movies_context SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [ ]:
# Validação da "casca de banana": nenhum documento nulo e nenhum filme perdido
ctx = spark.table("gold.gold_genai_movies_context")
nulos = ctx.filter(F.col("llm_context_document").isNull()).count()
assert nulos == 0, f"{nulos} documentos nulos no contexto da IA"
assert ctx.count() == dim_movies.count(), "Filmes da dim_movies ausentes na tabela de contexto"
display(ctx.limit(5))

## Desafio de Analytics
Recorte temporal das perguntas 5 e 6: a **data de referência** é a data de lançamento mais recente entre filmes
efetivamente lançados (`status = 'Lançado'` e data ≤ hoje), ignorando datas futuras/não lançadas.

**1. Receita total (R$) somada de todos os filmes da base**

In [ ]:
display(spark.sql("""
    SELECT format_number(SUM(receita_brl), 2) AS receita_total_brl,
           COUNT(receita_brl)                 AS filmes_com_receita
    FROM gold.fact_movies_performance
"""))

**2. Top 5 filmes por popularidade**

In [ ]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

**3. Quantidade de filmes por gênero (maior → menor)**

In [ ]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

**4. Top 10 filmes por receita, com ranking `RANK()`** (empates recebem a mesma posição)

In [ ]:
display(spark.sql("""
    WITH ranking AS (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT posicao, titulo, receita_usd, receita_brl
    FROM ranking
    WHERE posicao <= 10
    ORDER BY posicao
"""))

**5. Ator com mais participações em filmes lançados nos últimos 2 anos** (a partir da data de referência)

In [ ]:
display(spark.sql("""
    WITH referencia AS (
        SELECT MAX(data_lancamento) AS data_ref
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    ),
    filmes_periodo AS (
        SELECT m.sk_movie_id, r.data_ref
        FROM gold.dim_movies m CROSS JOIN referencia r
        WHERE m.status_filme = 'Lançado'
          AND m.data_lancamento >  add_months(r.data_ref, -24)
          AND m.data_lancamento <= r.data_ref
    ),
    contagem AS (
        SELECT p.nome_pessoa AS ator, COUNT(DISTINCT f.sk_movie_id) AS participacoes, MAX(f.data_ref) AS data_referencia
        FROM filmes_periodo f
        JOIN gold.bridge_movie_person b ON b.sk_movie_id = f.sk_movie_id
        JOIN gold.dim_people p ON p.sk_person_id = b.sk_person_id AND p.tipo_pessoa = 'Ator'
        GROUP BY p.nome_pessoa
    )
    SELECT * FROM (
        SELECT ator, participacoes, data_referencia, RANK() OVER (ORDER BY participacoes DESC) AS posicao
        FROM contagem
    ) WHERE posicao = 1           -- mostra todos os empatados na 1ª posição
"""))

**6. Produtora com maior lucro nos últimos 5 anos** (a partir da data de referência).
Filmes coproduzidos somam o lucro integral para cada produtora participante (não há percentual de participação na origem).

In [ ]:
display(spark.sql("""
    WITH referencia AS (
        SELECT MAX(data_lancamento) AS data_ref
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    ),
    filmes_periodo AS (
        SELECT m.sk_movie_id, r.data_ref
        FROM gold.dim_movies m CROSS JOIN referencia r
        WHERE m.status_filme = 'Lançado'
          AND m.data_lancamento >  add_months(r.data_ref, -60)
          AND m.data_lancamento <= r.data_ref
    ),
    lucro AS (
        SELECT c.nome_produtora,
               SUM(f.lucro_usd)              AS lucro_total_usd,
               SUM(f.lucro_brl)              AS lucro_total_brl,
               COUNT(DISTINCT f.sk_movie_id) AS filmes_com_lucro_calculado,
               MAX(p.data_ref)               AS data_referencia
        FROM filmes_periodo p
        JOIN gold.fact_movies_performance f ON f.sk_movie_id = p.sk_movie_id
        JOIN gold.bridge_movie_company b    ON b.sk_movie_id = p.sk_movie_id
        JOIN gold.dim_companies c           ON c.sk_company_id = b.sk_company_id
        WHERE f.lucro_usd IS NOT NULL
        GROUP BY c.nome_produtora
    )
    SELECT * FROM (
        SELECT *, RANK() OVER (ORDER BY lucro_total_usd DESC) AS posicao FROM lucro
    ) WHERE posicao = 1
"""))